# Forecasting with tabular foundational models

Tabular Foundation Models (FMs) are a new class of deep learning models that have been pre-trained on large amounts of tabular data so that they can be applied to a wide range of prediction tasks with minimal or no fine-tuning. These models have shown promising results in various domains, including forecasting. In this section, we will explore how to use tabular foundation models for forecasting tasks.



## Tabular Foundation Models vs. Machine Learning Models

Foundation models and traditional machine learning models approach forecasting in fundamentally different ways. Understanding these distinctions is crucial for knowing when and how to deploy each method.

**Zero-Shot Prediction**

Machine learning models require a training phase. You must fit the model on your historical target data (and exogenous variables) so the algorithm can learn the optimal weights and parameters for your specific time series. Foundation models are capable of zero-shot prediction. Because the model's parameters are already frozen from its massive pre-training phase, it can generate forecasts on your data immediately, without explicitly "learning" from your specific dataset first.

**The Role of the fit Method**

Calling .fit(y) in machine learning models triggers the optimization algorithm to minimize an error metric, updating the model's internal parameters. In foundation models, calling .fit(y) does not alter the model's neural network weights. Instead, the fit method acts as a standardizer. It is used merely to store necessary metadata (such as the last observed window of data, frequency, and scaling factors) so that the .predict() method has the exact context it needs to generate future values.

**Context Window vs. Engineered Lags**

Machine learning models rely on explicitly engineered features. In skforecast, you define lags or window_features to create a tabular dataset where past values are used as columns to predict the target. Foundation models rely on a context window. You pass a raw, sequential chunk of recent historical data (e.g., the last 512 observations) directly into the model at inference time. The attention mechanism inside the model automatically decides which past data points are most relevant.

## Tabular foundation models in skforecast

Foundational models that allow for regression tasks and follow the sklearn API (`fit` and `predict` methods) can be used with the `ForecasterRecursive`, `ForecasterRecursiveMultiSeries`, `ForecasterDirect` and `ForecasterDirectMultiVariate` classes. The following code snippets show how to use these models in skforecast.

<div class="admonition note" name="html-admonition" style="background: rgba(0,191,191,.1); padding-top: 0px; padding-bottom: 6px; border-radius: 8px; border-left: 8px solid #00bfa5; border-color: #00bfa5; padding-left: 10px; padding-right: 10px;">

<p class="title">
    <i style="font-size: 18px; color:#00bfa5;"></i>
    <b style="color: #00bfa5;">&#128161 Tip</b>
</p>

All these models can be run on a CPU; however, to unlock their full potential, it is highly recommended to use a GPU.
</div>

## Libraries and data

In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
path = str(Path.cwd().parent.parent)
print(path)
sys.path.insert(1, path)

import numpy as np
import pandas as pd
import skforecast

print(skforecast.__version__)

c:\Users\Joaquin\Documents\GitHub\skforecast
0.21.0


In [ ]:
# Libraries
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skforecast.datasets import fetch_dataset
from skforecast.recursive import ForecasterRecursive, ForecasterRecursiveMultiSeries
from skforecast.model_selection import (
    TimeSeriesFold,
    backtesting_forecaster,
    backtesting_forecaster_multiseries
)
from skforecast.plot import set_dark_theme
from tabicl import TabICLRegressor
import torch
color = '\033[1m\033[38;5;208m' 
print(f"{color}torch version: {torch.__version__}")
print("  Cuda available :", torch.cuda.is_available())
print("  Device name    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

torch version: 2.6.0+cu124
  Cuda available : True
  Device name    : NVIDIA RTX 2000 Ada Generation Laptop GPU


In [ ]:
# Data download
# ==============================================================================
data = fetch_dataset(name='vic_electricity')

# Aggregating in 1H intervals
# The Date column is eliminated so that it does not generate an error when aggregating.
data = data.drop(columns="Date")
data = (
    data
    .resample(rule="h", closed="left", label="right")
    .agg({
        "Demand": "mean",
        "Temperature": "mean",
        "Holiday": "mean",
    })
)
data.head(3)

╭──────────────────────────── vic_electricity ─────────────────────────────╮
│ Description:                                                             │
│ Half-hourly electricity demand for Victoria, Australia                   │
│                                                                          │
│ Source:                                                                  │
│ O'Hara-Wild M, Hyndman R, Wang E, Godahewa R (2022).tsibbledata: Diverse │
│ Datasets for 'tsibble'. https://tsibbledata.tidyverts.org/,              │
│ https://github.com/tidyverts/tsibbledata/.                               │
│ https://tsibbledata.tidyverts.org/reference/vic_elec.html                │
│                                                                          │
│ URL:                                                                     │
│ https://raw.githubusercontent.com/skforecast/skforecast-                 │
│ datasets/main/data/vic_electricity.csv                                   │
│                                                                          │
│ Shape: 52608 rows x 4 columns                                            │
╰──────────────────────────────────────────────────────────────────────────╯

,Demand,Temperature,Holiday
Time,,,
2011-12-31 14:00:00,4323.095350,21.225,1.0
2011-12-31 15:00:00,3963.264688,20.625,1.0
2011-12-31 16:00:00,3950.913495,20.325,1.0


In [4]:
# Split data into train-test
# ==============================================================================
data = data.loc['2012-01-01 00:00:00':'2014-12-30 23:00:00', :].copy()
end_train = '2014-11-30 23:59:00'
data_train = data.loc[: end_train, :].copy()
data_test  = data.loc[end_train:, :].copy()

print(f"Train dates: {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})")
print(f"Test dates : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})")


Train dates: 2012-01-01 00:00:00 --- 2014-11-30 23:00:00  (n=25560)
Test dates : 2014-12-01 00:00:00 --- 2014-12-30 23:00:00  (n=720)


### TabPFN-2.5

In [ ]:
# Create Forecaster
# ==============================================================================
forecaster = ForecasterRecursive(
                 estimator = TabICLRegressor(),
                 lags      = 15,
             )

In [ ]:
# Train Forecaster
# ==============================================================================
forecaster.fit(y=data_train["Demand"])
forecaster

In [ ]:
# Predictions: point forecast
# ==============================================================================
steps = 24
predictions = forecaster.predict(steps=steps, exog=data_test[["Temperature", "Holiday"]])
predictions.head(3)

In [ ]:
# Predictions: intervals
# ==============================================================================
predictions_intervals = forecaster.predict_interval(
    steps    = steps,
    exog     = data_test[["Temperature", "Holiday"]],
    interval = [10, 90],  # 80% prediction interval
)
predictions_intervals.head(3)

In [ ]:
# Backtesting
# ==============================================================================
cv = TimeSeriesFold(
        steps              = 24,
        initial_train_size = len(data.loc[:end_train]),
        refit              = False
)

metrics_levels, backtest_predictions = backtesting_forecaster(
    forecaster        = forecaster,
    series            = data['Demand'],
    exog              = data[["Temperature", "Holiday"]],
    cv                = cv,
    metric            = 'mean_absolute_error',
    suppress_warnings = True
)

print("Backtest metrics")
display(metrics_levels)
print("")
print("Backtest predictions")
backtest_predictions.head(4)

In [ ]:
# Plot predictions
# ==============================================================================
set_dark_theme()
fig, ax = plt.subplots(figsize=(7, 3))
data_test['Demand'].plot(ax=ax, label='test')
backtest_predictions['pred'].plot(ax=ax, label='predictions')
ax.legend();

<div class="admonition note" name="html-admonition" style="background: rgba(255,145,0,.1); padding-top: 0px; padding-bottom: 6px; border-radius: 8px; border-left: 8px solid #ff9100; border-color: #ff9100; padding-left: 10px; padding-right: 10px">

<p class="title">
    <i style="font-size: 18px; color:#ff9100; border-color: #ff1744;"></i>
    <b style="color: #ff9100;"> <span style="color: #ff9100;">&#9888;</span> Warning</b>
</p>

These results are for illustrative purposes. Since this is a widely available public dataset, it is highly probable that the model was exposed to these specific data points during its pre-training phase. As a result, the predictions may be more optimistic than what would be achieved in a real-world production environment.

</div>

With the class `ForecasterRecursiveMultiSeries` it is possible to model multiple time series at the same time.

In [ ]:
# Data
# ==============================================================================
data_multiseries = fetch_dataset(name="items_sales")
display(data_multiseries.head(3))

In [ ]:
# Split data into train-test
# ==============================================================================
end_train = '2014-07-15 23:59:00'
data_multiseries_train = data_multiseries.loc[:end_train, :]
data_multiseries_test  = data_multiseries.loc[end_train:, :]

In [ ]:
# Create and train Forecaster
# ==============================================================================
forecaster = ForecasterRecursiveMultiSeries(
                 estimator = TabICLRegressor(),
                 lags      = 15,
             )
forecaster.fit(series=data_multiseries_train)
forecaster

In [ ]:
# Predictions and prediction intervals
# ==============================================================================
steps = 24

# Predictions for item_1 and item_2
predictions_item_1 = forecaster.predict(steps=steps, levels=['item_1', 'item_2'])
display(predictions_item_1.head())
print("")

# Interval predictions for item_1 and item_2
predictions_intervals = forecaster.predict_interval(
    steps    = steps,
    levels   = ['item_1', 'item_2'],
    interval = [10, 90],  # 80% prediction interval
)
predictions_intervals.head()

In [ ]:
# Plot predictions
# ==============================================================================
set_dark_theme()
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(8, 5), sharex=True)

predictions = forecaster.predict(steps=len(data_multiseries_test), levels=data_multiseries.columns)
for i, col in enumerate(data_multiseries.columns):
    data_multiseries_train[col].plot(ax=axes[i], label='train')
    data_multiseries_test[col].plot(ax=axes[i], label='test')
    predictions.query(f"level == '{col}'").plot(ax=axes[i], label='predictions', color='white')
    axes[i].set_title(col)
    axes[i].set_ylabel('sales')
    axes[i].set_xlabel('')
    axes[i].legend(loc='upper left')

fig.tight_layout()
plt.show();